In [1]:
import numpy as np
import os
import tensorflow as tf

SEED = 31415

def set_seed(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # os.environ['TF_DETERMINISTIC_OPS'] = '1'
    
set_seed()

2025-02-04 14:42:04.461961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738680124.480288    4577 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738680124.485514    4577 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-04 14:42:04.503614: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Imports
import matplotlib.pyplot as plt
import warnings

from matplotlib import gridspec

plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')
warnings.filterwarnings("ignore") # to clean up output cells

In [3]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

BATCH_SIZE = 64
IMAGE_SIZE = [128, 128]

ds_train_ = image_dataset_from_directory(
    '../datasets/car-or-truck/train',
    labels='inferred',
    label_mode='binary',
    image_size=IMAGE_SIZE,
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
)
ds_valid_ = image_dataset_from_directory(
    '../datasets/car-or-truck/valid',
    labels='inferred',
    label_mode='binary',
    image_size=IMAGE_SIZE,
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=False,
)

def convert_to_float(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32)
    return image, label

AUTOTUNE = tf.data.experimental.AUTOTUNE
ds_train = (
    ds_train_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

ds_valid = (
    ds_valid_
    .map(convert_to_float)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

Found 5117 files belonging to 2 classes.


2025-02-04 14:42:08.718100: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 5051 files belonging to 2 classes.


In [4]:
from tensorflow import keras
from tensorflow.keras import layers

pretrained_base = tf.keras.models.load_model(
    '../resources/cv-course-models/vgg16-pretrained-base/vgg16-pretrained-base.h5',
)
pretrained_base.trainable = False

model = keras.Sequential([
    
    # Preprocessing
    layers.RandomFlip('horizontal'), # flip left-to-right
    layers.RandomContrast(0.5), # contrast change by up to 50%
    
    # Base
    pretrained_base,
    
    # Head
    layers.Flatten(),
    layers.Dense(6, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

In [ ]:
EPOCHS = 30

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=EPOCHS,
    verbose=1,
)

Epoch 1/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 712s 9s/step - binary_accuracy: 0.7179 - loss: 0.5521 - val_binary_accuracy: 0.8153 - val_loss: 0.4166
Epoch 2/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - binary_accuracy: 0.8357 - loss: 0.3790

In [ ]:
import pandas as pd

history_frame = pd.DataFrame(history.history)

history_frame.loc[:, ['loss', 'val_loss']].plot()
history_frame.loc[:, ['binary_accuracy', 'val_binary_accuracy']].plot()